in the textbook "Exercises in Latin Prosody", break down the questions on naming feet to a more straightforward format.

In [11]:
import json
from time import sleep

In [4]:
from openai import OpenAI


with open('openai-api-key.txt', 'r') as f:
    openai_api_key = f.read().strip()

client = OpenAI(api_key=openai_api_key)

In [8]:
with open('../data/semi_structured/prosody_aligned_text_blocks_backup.json', 'r') as f:
    data = json.load(f)

questions = data["PART III"]["CHAPTER I"]

In [9]:
PROMPT_TEMPLATE = """I will you give you the lines of a poem. Each line is divided into feet, separated by |. I will also give you a sentence or two that tells you the names of all the feet in the poem. Can you simplify this description, so that we have an explicit name for every single foot? It should be in this format:
First line: name of 1st foot, name of 2nd foot, ...
Second line: name of 1st foot, ...
...

Here is the poem:
{}

Here is the description of the feet:
{}
"""

In [47]:
responses = []


for i, question in enumerate(questions):
    print(f"Processing question {i+1} of {len(questions)}")
    #poem = question["question_text"]
    poem = fixed_question_texts[i]
    description = question["answer_text"]
    prompt = PROMPT_TEMPLATE.format(poem, description)

    response = client.responses.create(
        model="gpt-4o",
        input=prompt
    )

    #response_text = response.choices[0].message.content
    responses.append(
        {
            "question_text": poem,
            "answer_text": description,
            "raw_response": response
        }
    )

    sleep(1)


Processing question 1 of 6
Processing question 2 of 6
Processing question 3 of 6
Processing question 4 of 6
Processing question 5 of 6
Processing question 6 of 6


In [14]:
responses

[{'question_text': 'In the following exercises the pupil is required to mention the name of the feet, into which each verse is divided.\n1.\nInteger vitæ, scělě rīsquě | pūrŭs,\nNōn ělgēt Maurī jăcŭ līs, něļque arcŭ.',
  'answer_text': 'THE first, fourth, and fifth feet in each line are trochees, the second spondees, the third dactyls.',
  'raw_response': Response(id='resp_68713f2c5560819fa923ecc0d8bef4e509af5dc758ae816a', created_at=1752252204.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-2024-08-06', object='response', output=[ResponseOutputMessage(id='msg_68713f2cd594819fbc2f1121f8dd5ce409af5dc758ae816a', content=[ResponseOutputText(annotations=[], text="Certainly! Here's the simplified description:\n\n1.\nFirst line: Trochee, Spondee, Dactyl, Trochee\nSecond line: Trochee, Spondee, Dactyl, Trochee", type='output_text', logprobs=[])], role='assistant', status='completed', type='message')], parallel_tool_calls=True, temperature=1.0, tool_choice

In [48]:
clean_responses = []
for dict_ in responses:
    print(dict_['question_text'])
    print(dict_['answer_text'])
    print(dict_['raw_response'].output[0].content[0].text)
    print('----')
    dict_['response'] = dict_['raw_response'].output[0].content[0].text
    del dict_['raw_response']
    clean_responses.append(dict_)

with open('../data/semi_structured/prosody_feet_questions.json', 'w') as f:
    json.dump(clean_responses, f, indent=4)


"In the following exercises the pupil is required to mention the name of the feet, into which each verse is divided.
1.
Intĕ|gēr vī|tæ, scĕlĕ|rīsquĕ | pūrŭs,
Nōn ĕ|gēt Māu|rī jăcŭ|līs, nĕ|que ārcŭ."
THE first, fourth, and fifth feet in each line are trochees, the second spondees, the third dactyls.
Certainly! Here's the simplified description of the feet in the poem:

1.
Trochee, Spondee, Dactyl, Trochee, Trochee
Trochee, Spondee, Dactyl, Trochee, Trochee
----
"2.
Pătēr|nă rū|ră bō|būs ēx|ērcēt | sŭīs,
Sŏlū|tŭs ōm|nī fœ|nŏrē."
The fifth foot in the first line, and the third in the second line, are spondees, all the other feet are iambi.
Certainly! Here’s the simplified description:

First line: iamb, iamb, iamb, iamb, spondee, iamb  
Second line: iamb, iamb, spondee
----
"APPENDIX.

3.
Illi | rōbŭr ĕt æs | trĭplēx
Cīrcā | pēctŭs ĕrāt, | quī frăgĭlēm | trŭcī
Commi|sit pelago | ratem
Primus, | nec timuit | præcipitem A|fricum."
The first foot in each line is a spondee, the last an iambus

In [42]:
with open('../data/semi_structured/prosody_feet_questions.json', 'w') as f:
    json.dump(clean_responses, f, indent=4)

In [46]:
fixed_question_texts = [
    '''"In the following exercises the pupil is required to mention the name of the feet, into which each verse is divided.
1.
Intĕ|gēr vī|tæ, scĕlĕ|rīsquĕ | pūrŭs,
Nōn ĕ|gēt Māu|rī jăcŭ|līs, nĕ|que ārcŭ."''',
'''"2.
Pătēr|nă rū|ră bō|būs ēx|ērcēt | sŭīs,
Sŏlū|tŭs ōm|nī fœ|nŏrē."''',
'''"APPENDIX.

3.
Illi | rōbŭr ĕt æs | trĭplēx
Cīrcā | pēctŭs ĕrāt, | quī frăgĭlēm | trŭcī
Commi|sit pelago | ratem
Primus, | nec timuit | præcipitem A|fricum."''',
'''"4.
Quīd lătĕt, ūt | mărīnæ
Fīlĭūm dī|cūnt Thĕtĭdīs | sūb lăchrymō|să Trōjæ
Funera, ne | virilis
Cultus in cæ|dem et Lycias | proriperet | catervas ?"''',
'''"5.
Quī sĕrĕ|re īngĕnŭ|ūm vŏlĕt | ăgrŭm,
Liberat | arva pri|ùs fruti|cibus.
Dŏmĭnā|rĕ tŭmĭ|dūs; spī|rĭtūs | āltōs | gĕrē;
Sĕquĭtūr | sŭpēr|bōs ūl|tŏr ā | tērgō | Dĕūs."''',
'''"6.
Nĕquĕ pūgnō, | nĕquĕ sēgnī | pĕdĕ vīctūs,
Simul unctos | Tiberinis | humeros la|vit in undis
Hās cūm gĕmĭ|nă cōmpĕdĕ | dēdĭcāt că|tēnas
Saturne, ti|bi Zoïlus | annulos pri|ores."'''
]

In [40]:
for i in range(len(clean_responses)):
    clean_responses[i]['question_text'] = fixed_question_texts[i]

In [41]:
clean_responses

[{'question_text': '"In the following exercises the pupil is required to mention the name of the feet, into which each verse is divided.\n1.\nIntĕ|gēr vītæ, scĕlĕ|rīsquĕ | pūrŭs,\nNōn ĕ|gēt Māu|rī jăcŭ|līs, nĕ|que ārcŭ."',
  'answer_text': 'THE first, fourth, and fifth feet in each line are trochees, the second spondees, the third dactyls.',
  'response': "Certainly! Here's the simplified description:\n\n1.\nFirst line: Trochee, Spondee, Dactyl, Trochee\nSecond line: Trochee, Spondee, Dactyl, Trochee"},
 {'question_text': '"2.\nPătēr|nă rū|ră bō|būs ēx|ērcēt | sŭīs,\nSŏlū|tŭs ōm|nī fœ|nŏrē."',
  'answer_text': 'The fifth foot in the first line, and the third in the second line, are spondees, all the other feet are iambi.',
  'response': "Certainly! Here's the simplified description of the feet:\n\nFirst line: iamb, iamb, iamb, iamb, spondee\nSecond line: iamb, iamb, spondee"},
 {'question_text': '"APPENDIX.\n\n3.\nIlli | rōbŭr ĕt æs | trĭplēx\nCīrcā | pēctŭs ĕrāt, | quī frăgĭlēm | trŭc

In [59]:
def break_up_response(text):
    # break into list of strings 
    # ['First line: name of 1st foot, name of 2nd foot, ...', 'Second line: name of 1st foot, ...', ...]

    # split on newlines
    lines = text.split('\n')
    # remove empty strings
    lines = [line for line in lines if line.strip()]
    
    # only keep lines that start with 'First/second/third/fourth line:'
    new_lines = []
    for line in lines:
        found = False
        for start in ['first', 'second', 'third', 'fourth']:
            lower_line = line.lower()
            if lower_line.startswith(start + ' line:') or lower_line.startswith("line"):
                _, split_line = line.split(': ')
                new_lines.append(split_line.strip())
                found=True
                break 
            if found:
                break
    return new_lines

break_up_response(clean_responses[5]['response'])

['small ionic, small ionic, small ionic',
 'small ionic, small ionic, small ionic, small ionic',
 'great ionic, great ionic, dichoree, spondee',
 'great ionic, great ionic, dichoree, spondee']

In [43]:
def break_up_poem(poem):
    # only want text of the poem, no instructions or question
    # cheat by looking for lines that have | in them 
    lines = poem.split('\n')
    lines = [line for line in lines if '|' in line]
    return lines

break_up_poem(clean_responses[0]['question_text'])


['Intĕ|gēr vītæ, scĕlĕ|rīsquĕ | pūrŭs,',
 'Nōn ĕ|gēt Māu|rī jăcŭ|līs, nĕ|que ārcŭ."']

In [38]:
clean_responses[0]['question_text']

'In the following exercises the pupil is required to mention the name of the feet, into which each verse is divided.\n1.\nInteger vitæ, scělě rīsquě | pūrŭs,\nNōn ělgēt Maurī jăcŭ līs, něļque arcŭ.'

In [61]:
line_by_line_questions = []
question = "What is the name of each foot in the following line of poetry? Give your answer as a comma-separated list."
for i in range(len(clean_responses)):
    print('Question', i)
    poem_lines = break_up_poem(clean_responses[i]['question_text'])
    resp_lines = break_up_response(clean_responses[i]['response'])

    assert len(poem_lines) == len(resp_lines)

    for j in range(len(poem_lines)):
        print('  Line', j)
        q_id = f"{i+1}.{j+1}"
        question_text = question + '\n' + poem_lines[j]
        if len(resp_lines[j].split(',')) != len(poem_lines[j].split('|')):
            print(f'WARNING: mismatch for question {i} line {j}')
            print(poem_lines[j])
            print(resp_lines[j])

        line_by_line_questions.append({
            'source_name': "exercises-in-latin-prosody",
            'source_year': 1823,
            'question_id': "prosody_app.I." + q_id,
            'question_format': 'short_answer',
            'question_content': 'scansion',
            'difficulty': 'unknown',
            
            'question_language': 'english',
            'answer_language': 'english',
            
            'question': question_text,
            'multiple_choice_options': [],
            'answers': [resp_lines[j]]
        })

Question 0
  Line 0
  Line 1
Question 1
  Line 0
  Line 1
Sŏlū|tŭs ōm|nī fœ|nŏrē."
iamb, iamb, spondee
Question 2
  Line 0
  Line 1
  Line 2
  Line 3
Question 3
  Line 0
  Line 1
  Line 2
Funera, ne | virilis
Second Epitrit, Choriamb, Amphibrac
  Line 3
Question 4
  Line 0
  Line 1
  Line 2
  Line 3
Question 5
  Line 0
  Line 1
  Line 2
  Line 3


In [62]:
# save
with open('../data/structured/prosody_feet_questions.json', 'w') as f:
    json.dump(line_by_line_questions, f, indent=4)

In [63]:
len(line_by_line_questions)

20